In [1]:
from google.colab import files

uploaded = files.upload()

Saving credit_card_fraud_synthetic.csv to credit_card_fraud_synthetic.csv


In [2]:
import pandas as pd

df = pd.read_csv(
    "credit_card_fraud_synthetic.csv"
)

df.head()

,transaction_id,amount,transaction_hour,merchant_category,city,device_type,is_international,previous_transactions_24h,account_age_days,failed_attempts,is_fraud
0,1,2346.34,1,Shopping,Chennai,Tablet,0,15,1092,0,0
1,2,15050.61,7,Online,Mumbai,Mobile,0,6,285,4,0
2,3,6583.73,14,Shopping,Chennai,Mobile,0,18,230,0,0
3,4,4564.71,16,Online,Hyderabad,Tablet,0,5,545,2,0
4,5,848.12,5,Food,Hyderabad,Tablet,0,14,138,2,0


In [3]:
df.shape

(100000, 11)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

In [5]:
X = df.drop(
    "is_fraud",
    axis=1
)

y = df["is_fraud"]

In [6]:
X = pd.get_dummies(X)

X.head()

,transaction_id,amount,transaction_hour,is_international,previous_transactions_24h,account_age_days,failed_attempts,merchant_category_Electronics,merchant_category_Food,merchant_category_Grocery,...,merchant_category_Travel,city_Bangalore,city_Chennai,city_Delhi,city_Hyderabad,city_Mumbai,city_Pune,device_type_Laptop,device_type_Mobile,device_type_Tablet
0,1,2346.34,1,0,15,1092,0,False,False,False,...,False,False,True,False,False,False,False,False,False,True
1,2,15050.61,7,0,6,285,4,False,False,False,...,False,False,False,False,False,True,False,False,True,False
2,3,6583.73,14,0,18,230,0,False,False,False,...,False,False,True,False,False,False,False,False,True,False
3,4,4564.71,16,0,5,545,2,False,False,False,...,False,False,False,False,True,False,False,False,False,True
4,5,848.12,5,0,14,138,2,False,True,False,...,False,False,False,False,True,False,False,False,False,True


In [7]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42

)

In [8]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [9]:
model = RandomForestClassifier(

    n_estimators=100,

    random_state=42

)


model.fit(

    X_train,

    y_train

)

RandomForestClassifier(random_state=42)

In [10]:
prediction = model.predict(X_test)


print(
accuracy_score(
    y_test,
    prediction
)
)


print(
classification_report(
    y_test,
    prediction
)
)

0.94945
              precision    recall  f1-score   support

           0       0.95      1.00      0.97     19027
           1       0.27      0.02      0.04       973

    accuracy                           0.95     20000
   macro avg       0.61      0.51      0.51     20000
weighted avg       0.92      0.95      0.93     20000



In [11]:
import joblib


joblib.dump(

    model,

    "fraud_model.pkl"

)

['fraud_model.pkl']

In [12]:
!ls

credit_card_fraud_synthetic.csv  fraud_model.pkl  sample_data


In [13]:
%%writefile app.py


import streamlit as st
import pandas as pd
import joblib


model = joblib.load(
    "fraud_model.pkl"
)


st.title(
    "💳 Credit Card Fraud Detection System"
)


st.write(
    "Machine Learning based fraud prediction system"
)


amount = st.number_input(
    "Transaction Amount"
)


hour = st.number_input(
    "Transaction Hour"
)


international = st.selectbox(

    "International Transaction",

    [0,1]

)


failed = st.number_input(

    "Failed Attempts"

)



if st.button("Predict"):


    data = pd.DataFrame({

        "amount":[amount],

        "transaction_hour":[hour],

        "is_international":[international],

        "failed_attempts":[failed]

    })


    result = model.predict(data)


    if result[0] == 1:

        st.error(
            "🚨 Fraud Transaction Detected"
        )

    else:

        st.success(
            "✅ Normal Transaction"
        )

Writing app.py


In [14]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 70.2 MB/s eta 0:00:00


In [15]:
!streamlit run app.py &>/content/logs.txt &

In [21]:
!ngrok config add-authtoken 3F0KB12nNG4rVvsergLNiKhD0Jf_7HmJZXENAKodB4x3LZ4nZ

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [22]:
!cat /content/logs.txt



2026-06-26 08:36:57.876 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.135.20.92:8501



In [7]:
!ngrok config add-authtoken 3FfRWrEuxdNLKxVvxnpJjWaet6a_2w98hvLcowjh8xEvyqUL3


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [8]:
!ngrok config check

Valid configuration file at /root/.config/ngrok/ngrok.yml


In [9]:
!cat /content/logs.txt



2026-06-26 08:52:33.886 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.135.20.92:8501



In [10]:
!streamlit run app.py &>/content/logs.txt &

In [11]:
from pyngrok import ngrok

ngrok.kill()

url = ngrok.connect(8501)

print(url)

NgrokTunnel: "https://unpaired-traction-deception.ngrok-free.dev" -> "http://localhost:8501"


In [12]:
!cat /content/logs.txt



2026-06-26 08:56:40.711 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.135.20.92:8502



In [16]:
!pkill streamlit

In [17]:
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

In [18]:
!cat /content/logs.txt



2026-06-26 09:03:25.087 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.135.20.92:8501



In [19]:
from pyngrok import ngrok

ngrok.kill()

url = ngrok.connect(8501)

print(url)

NgrokTunnel: "https://unpaired-traction-deception.ngrok-free.dev" -> "http://localhost:8501"


In [20]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib


model = joblib.load("fraud_model.pkl")


st.title("💳 Credit Card Fraud Detection System")

st.write("Machine Learning based fraud prediction system")


# create 22 features same as training

features = []

for i in range(22):
    value = st.number_input(
        f"Feature {i+1}",
        value=0.0
    )

    features.append(value)



if st.button("Predict"):


    data = pd.DataFrame(
        [features]
    )


    prediction = model.predict(data)


    if prediction[0] == 1:

        st.error("🚨 Fraud Transaction Detected")

    else:

        st.success("✅ Normal Transaction")

Overwriting app.py


In [21]:
!pkill streamlit

In [22]:
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

In [23]:
from pyngrok import ngrok

ngrok.kill()

url = ngrok.connect(8501)

print(url)

NgrokTunnel: "https://unpaired-traction-deception.ngrok-free.dev" -> "http://localhost:8501"


In [25]:
from google.colab import files
files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
files.download("fraud_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
files.download("credit_card_fraud_synthetic.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>